# Naive RAG, Knowledge Graph, or Hybrid?

### A measured three-way comparison on a 3-page document

This notebook answers a question with evidence rather than assertion:

> **Which questions does each approach get wrong, and why?**

The corpus is `small.pdf`, a 3-page extract of the VIT FFCS Academic Regulations 4.0.
All three systems read the *same document* and answer the *same questions*.

**The measured result:**

| Question type | Naive RAG | Knowledge Graph | Hybrid |
|---|:---:|:---:|:---:|
| Local facts (negation, OR/AND logic, absence) | correct | correct | correct |
| Global aggregates (counts, exhaustive lists) | **WRONG** | correct | correct |
| Unmodelled prose (rationale, objectives, description) | correct | **NO DATA** | correct |

Neither pure approach wins. **They fail in opposite directions**, and that is precisely the
argument for hybrid — not a vague claim that "combining is better", but a specific, demonstrable
complementarity you can watch happen.

---

### What this notebook adds beyond the usual demo

1. **Every answer carries a verifiable citation** — page, passage, and a verbatim quote that is
   checked against the source text. Fabricated citations are detected and flagged.
2. **The knowledge graph is shown failing**, not just winning. Six questions the graph simply
   cannot answer, proven by searching every property in it.
3. **A real bug in the hybrid implementation is kept in**, because diagnosing it teaches more
   than a clean result would.

## 0. Setup

| Artefact | Built by | Contents |
|---|---|---|
| `chunks.json` | `chunk_pdf.py`, `add_citations.py` | 18 chunks with page/passage citations |
| `chroma_ffcs/` | `build_vectorstore.py` | 18 embeddings (BAAI/bge-small-en-v1.5) |
| Neo4j graph | `build_graph.py` | 126 nodes, 266 relationships |
| `naive_rag_results.json`, `hybrid_results.json` | recorded live runs | verbatim LLM answers |

Start the database first: `docker start ffcs-neo4j` (browser at http://localhost:7474).

In [1]:
# Make the workshop modules importable and data paths resolvable from anywhere.
import sys
from pathlib import Path

SRC = Path.cwd().parent / 'src' if (Path.cwd().parent / 'src').is_dir() else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
import json, re, textwrap

from pathlib import Path



from build_vectorstore import get_collection, get_model, embed_query


import config as C, kag, kg_coverage



chunks   = json.loads(C.CHUNKS_JSON.read_text())

by_id    = {c['id']: c for c in chunks['chunks']}

naive    = {r['id']: r for r in json.loads(C.NAIVE_RESULTS.read_text())}

hybrid   = json.loads(C.HYBRID_RESULTS.read_text())

naive_gaps  = {r['id']: r for r in hybrid['naive_on_gaps']}

hybrid_gaps = {r['id']: r for r in hybrid['hybrid_on_gaps']}

hybrid_agg  = {r['id']: r for r in hybrid['hybrid_on_aggregates']}



collection = get_collection()

driver, database = kag.connect()

with driver.session(database=database) as s:

    nodes = s.run("MATCH (n {demo:'ffcs_kg'}) RETURN count(n) AS n").single()['n']

    rels  = s.run("MATCH ()-[r {demo:'ffcs_kg'}]->() RETURN count(r) AS n").single()['n']



print(f"Document : {chunks['document']['pages']} pages, {len(chunks['chunks'])} chunks")

print(f"Chroma   : {collection.count()} vectors")

print(f"Neo4j    : {nodes} nodes, {rels} relationships")

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document : 3 pages, 18 chunks
Chroma   : 18 vectors
Neo4j    : 126 nodes, 266 relationships


## 1. Citations: page, passage, and a quote that is checked

An answer without a source is unusable in an academic setting, and a *fabricated* source is
worse than none, because it looks authoritative. So the pipeline does two things:

1. The LLM must return, for each source, the **chunk id and a verbatim sentence** copied from it.
2. The code **verifies** that the sentence really appears in that chunk. Anything that fails
   verification is reported as `UNVERIFIED`, never silently shown.

### Why "page N, passage M" and not "page N, paragraph M"

Paragraph numbering would be the natural citation unit, but **this PDF's paragraph structure does
not survive text extraction**. De-wrapping page 1 yields 23 fragments (each abbreviation is its
own short line) while page 3 yields only 2 (justified text, almost no short lines). Four different
page-3 chunks would all cite "paragraph 2" — a citation that cannot be followed.

So a citation here is: **page + passage position + verbatim quote**, with the paragraph number
added only on pages where it is trustworthy. The quote is what a reader actually uses to find the
claim; the page tells them where to look.

In [2]:
print('How each chunk cites itself:\n')
for c in chunks['chunks'][:6]:
    print(f"  {c['id']:<14} {c['citation']}")
print('  ...')
print(f"\nParagraph index reliable per page: {chunks['document']['paragraph_index_reliable']}")
print('(page 1 is False: de-wrapping split the abbreviation list into 23 one-line fragments)')

How each chunk cites itself:

  chunk_p1_01    page 1, passage 1 of 5
  chunk_p1_02    page 1, passage 2 of 5
  chunk_p1_03    page 1, passage 3 of 5
  chunk_p1_04    page 1, passage 4 of 5
  chunk_p1_05    page 1, passage 5 of 5
  chunk_p2_01    page 2, passage 1 of 7 (paragraphs 1-4)
  ...

Paragraph index reliable per page: {'1': False, '2': True, '3': True}
(page 1 is False: de-wrapping split the abbreviation list into 23 one-line fragments)


In [3]:
from cited_rag import show

# A recorded, cited answer from a live run. Note the verified quote under the answer.
show(naive_gaps['K1'])

Q: What do employers expect from students?
   retrieved: ['chunk_p2_02', 'chunk_p1_03', 'chunk_p2_01', 'chunk_p3_05']

   Employers expect students to have multi-disciplinary competency, leadership skills, and to be Information and Communication Technology (ICT) ready [chunk_p1_03].

   SOURCES (verified against the document):
     [page 1, passage 3 of 5]
       "Employers expect students to have multi-disciplinary competency, leadership skills, and be Information and Communication Technology (ICT) ready."


### The verifier catches invented quotes

This runs offline — no API call. We hand the verifier one genuine citation and one fabricated one
and watch it separate them. This is the guard that makes a citation worth printing.

In [4]:
from cited_rag import _normalise

def verify(chunk_id, quote):
    chunk = by_id.get(chunk_id)
    if chunk is None:
        return False, 'unknown chunk id'
    ok = _normalise(quote) in _normalise(chunk['text'])
    return ok, ('verified' if ok else 'quote not found in that passage')

tests = [
    ('chunk_p3_02', 'There is NO Entrance Examination.'),              # real
    ('chunk_p3_02', 'Fashion Technology applicants must sit VITEEE.'), # invented
    ('chunk_p3_06', 'based on a valid UCEED score or V-DAT'),          # real
    ('chunk_p9_99', 'anything at all'),                                # bad id
]
for cid, quote in tests:
    ok, reason = verify(cid, quote)
    print(f"  {'PASS' if ok else 'FAIL':<5} [{cid}] {reason:<34} \"{quote[:46]}\"")

  PASS  [chunk_p3_02] verified                           "There is NO Entrance Examination."
  FAIL  [chunk_p3_02] quote not found in that passage    "Fashion Technology applicants must sit VITEEE."
  PASS  [chunk_p3_06] verified                           "based on a valid UCEED score or V-DAT"
  FAIL  [chunk_p9_99] unknown chunk id                   "anything at all"


## 2. Where naive RAG fails: questions about the whole document

Six questions, answered wrongly by naive RAG in a live run. Five of them share one shape.

| # | Question | Naive RAG said | Truth |
|---|---|---|---|
| C13 | How many entrance exams are named? | **2** | 4 |
| C15 | Which schools are named? | **"not in the context"** | 2 schools |
| C7 | Facilities of the school offering B.Des | **generic prose, 0 of 5 correct** | 5 named facilities |
| C14 | List all 11 FFCS features | **fabricated 11 items** | features 0–10 |
| C9 | How many Academic Council meetings? | **9** | 10 |
| C5 | Which exam for M.Tech? | **VITEEE** | VITMEE |

> ### The shape: the answer is a property of the whole document, not of any single passage.

A count. An exhaustive list. No passage contains the answer, because the answer only exists once
you have seen **everything**.

In [5]:
for qid in ['C13', 'C15', 'C7', 'C14', 'C9', 'C5']:
    r = naive[qid]
    print('=' * 100)
    print(f"[{qid}] {r['q']}")
    print(f"  GROUND TRUTH : {r['gold']}")
    print(f"  retrieved    : {r['chunks']}")
    if r['evidence_chunks']:
        missed = [c for c in r['evidence_chunks'] if c not in r['chunks']]
        print(f"  evidence in  : {r['evidence_chunks']}"
              + (f'   <-- NEVER RETRIEVED: {missed}' if missed else ''))
    print('\n  NAIVE RAG ->')
    for line in textwrap.wrap(r['answer'], 92):
        print(f'      {line}')
    print('\n  KAG ->')
    with driver.session(database=database) as session:
        _, _, rows = kag.run(session, qid)
    for row in rows:
        for k, v in row.items():
            print(f'      {k:<22} {str(v)[:150]}')
    print()

[C13] How many distinct entrance examinations are named in the document? List them.
  GROUND TRUTH : 4: VITEEE, VITMEE, UCEED, V-DAT
  retrieved    : ['chunk_p3_02', 'chunk_p1_05', 'chunk_p1_01', 'chunk_p2_02']
  evidence in  : ['chunk_p3_01', 'chunk_p3_02', 'chunk_p3_03', 'chunk_p3_06']   <-- NEVER RETRIEVED: ['chunk_p3_01', 'chunk_p3_03', 'chunk_p3_06']

  NAIVE RAG ->
      There are 2 distinct entrance examinations named in the document: VITEEE, VITMEE.

  KAG ->
      n                      4
      exams                  ['VITEEE (VIT Engineering Entrance Exam)', "VITMEE (VIT Master's Entrance Exam)", 'UCEED (Undergraduate Common Entrance Exam for Design)', 'V-DAT (Des

[C15] Which schools are named in the document?
  GROUND TRUTH : V-SIGN (VIT School of Design) and VIT Business School
  retrieved    : ['chunk_p1_01', 'chunk_p1_02', 'chunk_p1_05', 'chunk_p2_03']
  evidence in  : ['chunk_p2_06', 'chunk_p3_04']   <-- NEVER RETRIEVED: ['chunk_p2_06', 'chunk_p3_04']

  NAIVE RAG ->
  

### Why this happens — the mechanism in four steps

**1. Top-k retrieval is sampling, not reading.** It returns the *k* passages most similar to the
question. That is right when the answer sits inside one passage. It is structurally wrong for an
aggregate, which needs *all* the evidence — and *k* is fixed before anyone knows how much
evidence exists.

**2. Similarity ranks each chunk independently, and aggregate questions give it no signal.** A
chunk contributing one item to a list of nine looks no more relevant than an off-topic chunk. In
C15, "school" is topically close to the education vocabulary saturating page 1, so retrieval
returned page-1 chunks about abbreviations. Neither school is named there. Retrieval did its job;
similarity was simply the wrong signal.

**3. The LLM cannot tell a complete context from an incomplete one.** This is the crux. Nothing
in the prompt says *"you have been shown the entire document"*. The model sees four passages and
no sign that fourteen others exist, so it answers from what it has — fluently, without hedging.
**Naive RAG cannot distinguish "the answer is 2" from "I was shown 2 of the 4".**

**4. When the question presupposes a count, the model manufactures one.** C14 is the sharpest
case: asked to *"list all 11 features"* with two of four relevant chunks in context, it produced
exactly 11 items by splitting single features apart and inventing others.

### Why the graph is right by construction

The graph did the global pass **once, at build time**. Reading the whole document is not skipped
in a KG pipeline — it is *front-loaded*. By query time the entities form a **closed set**:

```cypher
MATCH (x:EntranceExam) RETURN count(x)    // 4. Not a sample. The set.
```

> **A vector store is a local index and answers local questions. A knowledge graph is a global
> index and answers global questions.** The failures above are a category error — asking a
> sampling system for a census.

**C5 is a different animal.** The source says students are admitted "based on their **VITEEE**
ranks" in a paragraph otherwise about VITMEE — a typo in the original. Naive RAG faithfully
repeated it; the graph holds the curated fact and keeps the original wording in a `source_note`.
State this plainly: **the graph is right here because a human corrected the source while building
it.** That is a real advantage — a KG is where curation lives and can be audited — but it is not
magic and it is not free.

In [6]:
# Evidence for step 2: the graph can tell us where the answer to C13 actually lives -
# a question the vector store cannot answer about itself.
with driver.session(database=database) as session:
    spread = {r['exam']: r['chunks'] for r in session.run('''
        MATCH (x:EntranceExam)-[:MENTIONED_IN]->(c:Chunk)
        RETURN x.name AS exam, collect(c.id) AS chunks ORDER BY exam''')}

retrieved = naive['C13']['chunks']
print('Q: How many distinct entrance examinations are named in the document?\n')
print(f'Retrieved at k=4: {retrieved}\n')
needed = set()
for exam, cs in spread.items():
    needed.update(cs)
    seen = [c for c in cs if c in retrieved]
    print(f"  {exam:<8} appears in {str(cs):<46} {'VISIBLE' if seen else 'INVISIBLE to the LLM'}")
print(f'\nChunks needed for a complete answer: {len(needed)}  ->  {sorted(needed)}')
print(f'Chunks the LLM actually saw        : {len(retrieved)}  ->  {sorted(retrieved)}')
print(f'Overlap                            : {len(set(needed) & set(retrieved))} of {len(needed)}')
print()
print('Note the trap: k=4 was NUMERICALLY enough - four chunks would have sufficed.')
print('Retrieval simply chose four different ones. Raising k is not a reliable fix,')
print('because you never know in advance how much evidence a question needs, nor')
print('whether similarity will rank that evidence highly.')

Q: How many distinct entrance examinations are named in the document?

Retrieved at k=4: ['chunk_p3_02', 'chunk_p1_05', 'chunk_p1_01', 'chunk_p2_02']

  UCEED    appears in ['chunk_p3_06']                                INVISIBLE to the LLM
  V-DAT    appears in ['chunk_p3_06']                                INVISIBLE to the LLM
  VITEEE   appears in ['chunk_p3_03', 'chunk_p3_02', 'chunk_p3_01']  VISIBLE
  VITMEE   appears in ['chunk_p3_02', 'chunk_p3_03']                 VISIBLE

Chunks needed for a complete answer: 4  ->  ['chunk_p3_01', 'chunk_p3_02', 'chunk_p3_03', 'chunk_p3_06']
Chunks the LLM actually saw        : 4  ->  ['chunk_p1_01', 'chunk_p1_05', 'chunk_p2_02', 'chunk_p3_02']
Overlap                            : 1 of 4

Note the trap: k=4 was NUMERICALLY enough - four chunks would have sufficed.
Retrieval simply chose four different ones. Raising k is not a reliable fix,
because you never know in advance how much evidence a question needs, nor
whether similarity will rank th

## 3. Being honest: where naive RAG did *not* fail

A demo where one side wins everything is a demo that was rigged, and students can smell it.
Naive RAG answered **10 of 18** correctly — including all three failure modes the textbooks
attribute to it. Show this **before** showing the failures.

In [7]:
for r in naive.values():
    if r['verdict'] == 'CORRECT':
        print(f"  CORRECT  {r['id']:<5} {r['q'][:74]}")
print()
print('These include the three failure modes usually claimed for vector RAG:')
print('  C1  explicit negation   - "there is NO entrance examination"  -> answered correctly')
print('  C2  OR/AND logical form - "UCEED or V-DAT, plus 10+2"         -> answered correctly')
print('  C3  absent information  - "M.Des route is not stated"         -> answered correctly')
print()
print('All three are LOCAL questions: the answer sits in one passage, retrieved at rank 1.')
print('A capable LLM then reads it correctly. The textbook story is out of date for')
print('strong models on small corpora.')

  CORRECT  C1    Which entrance exam must a B.Tech Fashion Technology applicant take?
  CORRECT  C1b   List every B.Tech programme and state the entrance exam each one requires.
  CORRECT  C2    Do I need both a UCEED score and a V-DAT score to get into B.Des?
  CORRECT  C2b   I have a UCEED score but no V-DAT score. Am I eligible for B.Des?
  CORRECT  C3    How is a student admitted to M.Des?
  CORRECT  C3b   What score does M.Des require?
  CORRECT  C6    How many FFCS regulation versions came before version 4.0? List them in or
  CORRECT  C8    Which programmes in this document require no entrance examination at all?
  CORRECT  C12   Which course basket has no abbreviation defined in the document?
  CORRECT  C17   Is a V-DAT score accepted for admission to M.Des?

These include the three failure modes usually claimed for vector RAG:
  C1  explicit negation   - "there is NO entrance examination"  -> answered correctly
  C2  OR/AND logical form - "UCEED or V-DAT, plus 10+2"         ->

## 4. Where the *knowledge graph* fails

Now the other direction, which most demos leave out.

A knowledge graph contains exactly what its ontology chose to model. Everything else in the
document — rationale, objectives, qualitative description, incidental detail — **does not exist**
in the graph. No Cypher query can retrieve it, because it was never extracted.

### A fair test

Rather than writing one query per question and declaring failure when it returns nothing, the
cell below searches **every property of every node and relationship** in the graph for the terms
the answer needs, and separately checks whether those terms appear in the chunk text. If the
content is in the text but nowhere in the graph, no query could ever have found it.

In [8]:
with driver.session(database=database) as session:
    print(f"{'id':<5} {'question':<58} {'in graph?':<12} {'in text?'}")
    print('-' * 100)
    for cid, question, terms in kg_coverage.CASES:
        graph, text = kg_coverage.coverage(session, terms)
        print(f"{cid:<5} {question[:56]:<58} "
              f"{('YES (' + str(len(graph)) + ')') if graph else 'NO':<12} "
              f"{('YES ' + str([t['chunk'] for t in text])) if text else 'NO'}")

id    question                                                   in graph?    in text?
----------------------------------------------------------------------------------------------------


K1    What do employers expect from students?                    NO           YES ['chunk_p1_03']
K2    What is the main objective of the design programmes at V   NO           YES ['chunk_p3_04']
K3    How are V-SIGN students trained to approach product desi   NO           YES ['chunk_p3_05']
K4    What kind of building houses the design facilities?        NO           YES ['chunk_p3_04']
K5    Why does VIT say present-day students need a flexible sy   NO           YES ['chunk_p1_03']
K6    Does being called for counselling guarantee admission to   YES (1)      YES ['chunk_p3_02']
K7    Whose interpretation is final and binding in a dispute o   NO           YES ['chunk_p2_07']


K8    What is VTOP used for?                                     YES (1)      YES ['chunk_p1_03']
K9    What helps slow learners under FFCS?                       YES (1)      YES ['chunk_p2_01']
K10   What does the document say about research for UG student   YES (1)      YES ['chunk_p2_01']


Six questions — **K1–K5 and K7** — are in the document but absent from the graph. Here is what
naive RAG does with them, and what the graph offers instead.

In [9]:
for qid in ['K1', 'K2', 'K3', 'K4', 'K5', 'K7']:
    r = naive_gaps[qid]
    print('=' * 100)
    print(f"[{qid}] {r['question']}")
    print(f"  GROUND TRUTH : {r['gold']}")
    print(f"  NAIVE RAG    : {r['answer'][:200]}")
    for c in r['citations']:
        print(f"                 [{c['citation']}] \"{c['quote'][:76]}\"")
    print(f"  KAG          : no such entity or property in the graph - nothing to query")
    print()

[K1] What do employers expect from students?
  GROUND TRUTH : Multi-disciplinary competency, leadership skills, ICT ready.
  NAIVE RAG    : Employers expect students to have multi-disciplinary competency, leadership skills, and to be Information and Communication Technology (ICT) ready [chunk_p1_03].
                 [page 1, passage 3 of 5] "Employers expect students to have multi-disciplinary competency, leadership "
  KAG          : no such entity or property in the graph - nothing to query

[K2] What is the main objective of the design programmes at V-SIGN?
  GROUND TRUTH : Create a new breed of problem solvers in industrial design.
  NAIVE RAG    : The main objective of the programmes is to create a new breed of problem solvers in the domain of industrial design [chunk_p3_04].
                 [page 3, passage 4 of 6 (paragraph 2)] "The main objective of the programmes is to create a new breed of problem sol"
  KAG          : no such entity or property in the graph - nothing to qu

### Why the graph fails — an ontology is a lossy projection

Building a knowledge graph means deciding, in advance, **which kinds of things matter**. Our
ontology models programmes, exams, regulations, meetings, facilities. It does not model:

- *rationale* — why VIT introduced a flexible credit system (K5)
- *objectives* — what the design school is trying to achieve (K2)
- *pedagogy* — how students are trained to approach design (K3)
- *expectations* — what employers want from graduates (K1)
- *incidental description* — that the facilities sit in "a sprawling new building" (K4)
- *legal nuance* — whose interpretation is "final and binding" in a dispute (K7)

None of this is an oversight to be patched. It is the **nature of the technique**: extraction
selects, and whatever is not selected is discarded. You could extend the ontology to cover these
six, and a student would immediately find six more. The text always holds more than any schema
anticipates.

> **Naive RAG loses completeness. A knowledge graph loses everything the schema did not
> anticipate.** The text keeps it all — which is why the text has to stay in the system.

### A caveat worth flagging

Four questions in the coverage table (K6, K8, K9, K10) *are* answerable from the graph — but look
at **why**: they are answerable because `Feature` nodes carry the full sentence in a `text`
property, and because we attached free-text `note` properties to some nodes. In other words, the
graph answers those by **storing unstructured text inside itself**, not by modelling structure.

That is a legitimate and common design, and it is worth being honest with students that it is a
halfway house: the moment you store prose in a node property, you have rebuilt a small, worse
document store inside your graph.

## 5. Hybrid RAG

The two systems fail in opposite directions, so the combination is not merely additive — each one
covers precisely the other's blind spot.

### Design: three blocks of context

| Block | Source | Fixes |
|---|---|---|
| `PASSAGES` | top-k chunks + citations | prose, rationale, nuance — the KG's blind spot |
| `GRAPH FACTS` | 1-hop neighbourhood | structure spanning chunk boundaries |
| `COMPLETE SETS` | every member of a relevant type | counts and exhaustive lists — naive RAG's blind spot |

`COMPLETE SETS` is the block that repairs the counting failures. If entrance exams are relevant,
the graph supplies **all four**, not just those that happened to be retrieved. That is a census,
and only the graph can produce one.

### The design detail that matters most: enter the graph from *two* directions

```
  (a) from the retrieved chunks   - entities the passages mention
  (b) from the question itself    - entity names and TYPE words ("schools" -> :School)
```

Direction **(b)** is not a refinement, it is essential — and I learned that by getting it wrong.
See the next section.

In [10]:
from hybrid_rag import labels_from_question, build_context

for q in ['Which schools are named in the document?',
          'List all the facilities at the school that offers B.Des.',
          'How many distinct entrance examinations are named?',
          'What do employers expect from students?']:
    print(f'{q[:56]:<58} -> types asked about: {labels_from_question(q)}')
print()
print('Note the last one returns [] - a prose question adds no graph noise.')

with driver.session(database=database) as session:
    ids, context, facts, census = build_context(
        session, 'Which schools are named in the document?', get_model(), collection)
print(f'\nContext for the schools question: {len(ids)} passages, {len(facts)} graph facts, {len(census)} complete sets')
print('\nCOMPLETE SETS block:')
for c in census:
    print(textwrap.indent(textwrap.fill(c, 92), '    '))

Which schools are named in the document?                   -> types asked about: ['School']
List all the facilities at the school that offers B.Des.   -> types asked about: ['School', 'Facility']
How many distinct entrance examinations are named?         -> types asked about: ['EntranceExam']
What do employers expect from students?                    -> types asked about: []

Note the last one returns [] - a prose question adds no graph noise.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10371.36it/s]


Context for the schools question: 4 passages, 71 graph facts, 5 complete sets

COMPLETE SETS block:
    ALL CouncilMeeting in the document (10 total): 18th Academic Council meeting, 20th Academic
    Council meeting, 27th Academic Council meeting, 28th Academic Council meeting, 37th Academic
    Council meeting, 46th Academic Council meeting, 59th Academic Council meeting, 71st Academic
    Council meeting, 72nd Academic Council meeting, Standing Committee meeting of the Academic
    Council
    ALL CourseBasket in the document (9 total): Ability Enhancement Courses, Discipline Core,
    Discipline Elective, Discipline Linked Engineering Courses, Foundation Core, Open Elective,
    Project and Internship, Skill Enhancement Courses, Specialization Elective
    ALL GoverningBody in the document (4 total): Academic Council, Academic Policy Committee,
    Management, Standing Committee of the Academic Council
    ALL Regulation in the document (9 total): B.Tech. Degree Programme Regulatio

### A bug worth keeping in the notebook

The first version of the hybrid pipeline **failed C7 and C15 anyway**, even though the graph held
both answers. Two independent causes, both instructive:

**1. A silent truncation.** The code capped the fact list at `facts[:60]`. Facts were sorted
alphabetically by label, so `School -HAS_FACILITY-> PROTICS Studio` sorted near the end and was
cut. The context looked full, the answer looked plausible, and the evidence was simply gone.
*Never truncate an ordered list at an arbitrary boundary — rank by relevance, or keep it all.*

**2. Seeding only from retrieved chunks.** Graph expansion started from entities mentioned in the
retrieved passages. For C15, retrieval returned page-1 chunks that mention no school — so the
graph was **never asked about schools at all**. Hybrid had silently inherited naive RAG's
retrieval failure.

> A hybrid that enters the graph only through retrieved chunks is not really a hybrid. It is
> vector RAG with extra steps, and it fails exactly where vector RAG fails.

Reading type words out of the question fixes this: *"which **schools**..."* → `:School` → the
census of all schools, regardless of what retrieval returned.

In [11]:
print('HYBRID on the questions NAIVE RAG got wrong (global aggregates)')
for qid, r in hybrid_agg.items():
    print('=' * 100)
    print(f"[{qid}] {r['question']}")
    print(f"  GROUND TRUTH : {r['gold']}")
    print(f"  NAIVE RAG    : {naive[qid]['answer'][:150]}")
    print(f"  HYBRID       : {r['answer'][:260]}")
    print(f"  context      : {r['n_facts']} graph facts, {r['n_census']} complete sets")
    print()

HYBRID on the questions NAIVE RAG got wrong (global aggregates)
[C13] How many distinct entrance examinations are named in the document? List them.
  GROUND TRUTH : 4: VITEEE, VITMEE, UCEED, V-DAT
  NAIVE RAG    : There are 2 distinct entrance examinations named in the document: VITEEE, VITMEE.
  HYBRID       : There are 4 distinct entrance examinations named in the document: UCEED, V-DAT, VITEEE, and VITMEE. Dates for exams like VITEEE and VITMEE are announced through the university website or media [chunk_p3_02].
  context      : 63 graph facts, 5 complete sets

[C15] Which schools are named in the document?
  GROUND TRUTH : VIT School of Design (V-SIGN) and VIT Business School
  NAIVE RAG    : The context does not contain the answer.
  HYBRID       : The document names two schools: VIT Business School and VIT School of Design.
  context      : 71 graph facts, 5 complete sets

[C7] List all the facilities at the school that offers B.Des.
  GROUND TRUTH : PROTICS Studio, 3D-iD Studio,

In [12]:
print('HYBRID on the questions the KNOWLEDGE GRAPH cannot answer (unmodelled prose)')
for qid in ['K1', 'K2', 'K3', 'K4', 'K5', 'K7']:
    r = hybrid_gaps[qid]
    print('=' * 100)
    print(f"[{qid}] {r['question']}")
    print(f"  GROUND TRUTH : {r['gold']}")
    print(f"  KAG          : no such data in the graph")
    print(f"  HYBRID       : {r['answer'][:220]}")
    for c in r['citations'][:2]:
        print(f"                 [{c['citation']}] \"{c['quote'][:70]}\"")
    print()

HYBRID on the questions the KNOWLEDGE GRAPH cannot answer (unmodelled prose)
[K1] What do employers expect from students?
  GROUND TRUTH : Multi-disciplinary competency, leadership skills, ICT ready.
  KAG          : no such data in the graph
  HYBRID       : Employers expect students to have multi-disciplinary competency, leadership skills, and be Information and Communication Technology (ICT) ready [chunk_p1_03].
                 [page 1, passage 3 of 5] "Employers expect students to have multi-disciplinary competency, leade"

[K2] What is the main objective of the design programmes at V-SIGN?
  GROUND TRUTH : Create a new breed of problem solvers in industrial design.
  KAG          : no such data in the graph
  HYBRID       : The main objective of the V-SIGN programmes is "to create a new breed of problem solvers in the domain of industrial design" [chunk_p3_04]. Additionally, the programmes aim to develop students' skills, knowledge, and apt
                 [page 3, passage 4 of 

## 6. Summary

### The three-way result, measured

| Question | Naive RAG | KAG | Hybrid |
|---|:---:|:---:|:---:|
| C1 Which exam for Fashion Technology? | correct | correct | correct |
| C2 UCEED *and* V-DAT for B.Des? | correct | correct | correct |
| C3 How is M.Des admitted? | correct | correct | correct |
| C13 How many entrance exams? | **wrong (2)** | correct | **correct (4)** |
| C15 Which schools are named? | **wrong (none)** | correct | **correct (2)** |
| C7 Facilities of the B.Des school | **wrong** | correct | **correct (5)** |
| C9 How many Council meetings? | **wrong (9)** | correct | **correct (10)** |
| C14 List all 11 FFCS features | **wrong (fabricated)** | correct | correct |
| C5 Which exam for M.Tech? | **wrong (VITEEE)** | correct | correct |
| K1 What do employers expect? | correct | **no data** | correct |
| K2 Objective of the design programmes | correct | **no data** | correct |
| K3 How are design students trained? | correct | **no data** | correct |
| K4 What kind of building? | correct | **no data** | correct |
| K5 Why do students need flexibility? | correct | **no data** | correct |
| K7 Whose interpretation is binding? | correct | **no data** | correct |

### Three sentences worth memorising

1. **Top-k retrieval is sampling.** It answers questions about a passage, never about a corpus,
   because it never sees the corpus.
2. **An ontology is a lossy projection.** A graph holds what you chose to model, and the document
   always contains more than any schema anticipated.
3. **The LLM cannot tell incomplete evidence from complete evidence**, so it answers both with
   the same confidence. Every failure in this notebook is fluent and none is hedged.

### When to use which

| Approach | Use when | Fails when |
|---|---|---|
| Naive RAG | large corpus, local questions, no budget to model the domain | the question is about the whole corpus |
| Knowledge graph / KAG | counting, enumerating, auditable and curated answers | the question is about something you never modelled |
| **Hybrid** | you need both, and can afford to build and maintain both | — but only if the graph is entered from the question, not just from retrieved chunks |

### Methodology and caveats — state these out loud

**The corpus is 3 pages.** Retrieval is nearly solved at this scale; a student will rightly note
that the whole document fits in one prompt. That objection is correct and it *sharpens* the claim:
the graph's advantage here is **completeness and curation, not recall**. Scale to 3,000 pages and
local questions begin to fail too — but the global questions were already failing at three pages,
for a reason no better embedding will fix.

**Answers came from more than one model.** Free-tier daily quotas ran out mid-experiment, so runs
used `gemini-2.5-flash`, `gemini-flash-latest`, and `gemini-3-flash-preview`. Each recorded result
stores the model that produced it. The conclusions do not rest on any single model: the aggregate
failures reproduced across model families, and the structural evidence — *which* chunks were
retrieved versus which chunks hold the answer — is model-independent.

**Recorded, not live.** Answers are frozen from real runs so the notebook opens fully populated.
Set `live=True` in `cited_rag` / `hybrid_rag` to reproduce them, mindful of the 5 requests/minute
free-tier limit.

In [13]:
# Reproduce any answer live (needs quota):
#   from cited_rag import cited_rag, show; show(cited_rag('Which schools are named?'))
#   from hybrid_rag import hybrid_rag, show; show(hybrid_rag('Which schools are named?'))
driver.close()
print('Done.')

Done.
